In [ ]:
#| default_exp search

# search

> Web search across several engines, merging and reranking the results, and research digests

`search` sends one query to several search engines and merges their results into one list. Each engine finds pages that other engines miss. A page that several engines rank highly usually gives a useful result. Fossick uses [ddgs](https://github.com/deedy5/ddgs) to reach the engines. `ddgs` does not need API keys. `search` merges the lists with reciprocal rank fusion (RRF). It can then reorder the merged list by query relevance.

`google` loads the Google results page in a stealth browser. It is slower than `search`. `research` searches, downloads readable results, and returns a markdown digest with a citation for each source. You can pass the digest to a language model.

In [ ]:
#| export
import json, os, re, sys
from math import log
from urllib.parse import quote_plus, urlsplit, urlunsplit, parse_qsl, urlencode
from fastcore.all import first, bind, ifnone
from fastcore.parallel import parallel
from scrapling import Selector
from fossick.core import to_md, html2md, fetch, fossick_cache, _blocked, search_yt, _kept, _keep
from fossick.quality import norm_url, curate as _curate, host, plan as _plan, interleave

## The ddgs client

`get_ddgs` returns a shared `DDGS` client for each timeout and proxy setting. It creates each client on first use. `ddgs_env` reads proxy and certificate settings from the environment. This supports search behind a corporate proxy. It reads the proxy from `DDGS_PROXY`, `HTTPS_PROXY` or `https_proxy`. It reads a certificate bundle from `FOSSICK_CA_BUNDLE`, `SSL_CERT_FILE`, `REQUESTS_CA_BUNDLE` or `CURL_CA_BUNDLE`.

In [ ]:
#| export
_warned = set()
def _warn(key:str, msg:str):
    "Print `msg` to stderr once per process. A warning is better than an empty result with no explanation."
    if key not in _warned:
        _warned.add(key)
        print(f'fossick: {msg}', file=sys.stderr)

def ddgs_env() -> dict:
    "Return proxy and certificate settings for ddgs, read from the environment."
    ev = lambda *ks: first(os.environ.get(k) for k in ks if os.environ.get(k))
    ca = ev('FOSSICK_CA_BUNDLE', 'SSL_CERT_FILE', 'REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE')
    return dict(proxy=ev('DDGS_PROXY', 'HTTPS_PROXY', 'https_proxy'), verify=ca or True)

_ddgs = {}
def get_ddgs(timeout:int=10):
	"Return the shared ddgs client for these timeout, proxy and certificate settings."
	env = ddgs_env()
	if (key := (timeout, *sorted(env.items()))) not in _ddgs:
		try: from ddgs import DDGS
		except ImportError as e: raise ImportError('web search needs ddgs: pip install ddgs') from e
		_ddgs[key] = DDGS(timeout=timeout, **env)
	return _ddgs[key]

## Asking each engine

For text search, Fossick queries each engine in `TEXT_BACKENDS` separately. It keeps each engine's results in that engine's rank order. It does not use `ddgs` combined search. That search picks engines at random and favours Wikipedia. `backends` lists every `ddgs` engine for a category. When `pages` is greater than 1, Fossick asks each engine for that many result pages. It drops results on later pages when they repeat earlier results. The test checks that ranks continue across pages.

In [ ]:
#| export
TEXT_BACKENDS = ('google', 'brave', 'duckduckgo', 'mojeek', 'yahoo', 'startpage')

def backends(category:str='text') -> list:
    "Return the names of every ddgs engine available for `category`."
    from ddgs.engines import ENGINES
    return sorted(ENGINES.get(category, {}))

def _engine_hits(be:str, q:str, region='us-en', page=1, timeout=8, tries=2, **kw) -> list:
    "Return one backend's results in rank order with `engine` and `rank`. Pass `kw` to ddgs."
    from ddgs.engines import ENGINES
    cls = ENGINES['text'].get(be)
    if cls is None: return []
    for _ in range(tries):
        try:
            hits = ifnone(cls(timeout=timeout, **ddgs_env()).search(q, region=region, page=page, **kw), [])
            if hits: return [dict(h.__dict__, engine=be, rank=i+1) for i, h in enumerate(hits)]
        except Exception: pass
    return []

def _backend_lists(q:str, backends, region='us-en', timeout=8, pages=1, page=1, **kw) -> list:
    "Return one rank-ordered list per backend from `pages` result pages. Start at `page`. Run each backend and page in parallel."
    jobs = [(be, pg) for be in backends for pg in range(page, page+pages)]
    if not jobs: return []
    got = parallel(lambda j: (j, _engine_hits(j[0], q, region=region, page=j[1], timeout=timeout, **kw)),
                   jobs, n_workers=len(jobs), threadpool=True)
    by_be = {}
    for (be, pg), hits in got: by_be.setdefault(be, {})[pg] = hits
    out = []
    for be in backends:
        merged, seen = [], set()
        for pg in sorted(by_be.get(be, {})):
            for h in by_be[be][pg]:
                if (key := norm_url(h.get('href') or h.get('url') or '')) in seen: continue  # Later pages can repeat earlier pages.
                seen.add(key)
                merged.append(dict(h, rank=len(merged)+1))    # Keep one rank sequence across pages.
        if merged: out.append(merged)
    return out


## Merging the lists

Reciprocal rank fusion (RRF) merges ranked lists without using the engines' own scores. The scores are not comparable. Each page gets `1/(k + rank)` from each list that contains it. RRF adds these values. A page ranked first scores more than a page ranked eighth. A page found by several engines gets one value from each engine. `k` is 60 by default. It controls how much more the top ranks count than the lower ranks.

`rrf` treats two URLs as the same page when they differ only in scheme, `www.`, a trailing slash or tracking parameters. It uses `norm_url` from `fossick.quality`. For each page, it keeps the longest snippet. It also keeps the fused `score` and `engines`, which lists the engines that found the page.

In [ ]:
#| export
def rrf(lists:list,      # Ranked result lists, one per backend.
        k:int=60,        # RRF damping constant.
       ) -> list:
    "Merge ranked lists with reciprocal rank fusion. Score each URL with the sum of `1/(k+rank)` across its lists."
    score, best, engines = {}, {}, {}
    for hits in lists:
        for i, h in enumerate(hits):
            if not (key := norm_url(h.get('href') or h.get('url') or '')): continue
            score[key] = score.get(key, 0.0) + 1/(k + h.get('rank', i+1))
            engines.setdefault(key, []).append(h.get('engine', ''))
            cur = best.get(key)                                  # Keep the longest available snippet.
            if cur is None or len(h.get('body') or '') > len(cur.get('body') or ''): best[key] = h
    out = sorted(score, key=lambda u: -score[u])
    return [dict(best[u], score=round(score[u], 6), engines=[e for e in engines[u] if e]) for u in out]


## Reranking

Search engines use more than exact query words when they rank results. `rerank` reorders results by how well their title, snippet and URL match the query. It combines this relevance score with the merged score. `mix` sets the weight of the merged score. Its default is 0.4. A value of 1 ignores the reranker. When `method` is not `'none'`, `rerank` adds a `rerank` field with the relevance score.

`method` selects the scorer. `'lexical'` uses BM25, a standard word-matching formula. `bm25` computes scores from the result list. It does not need a stored index. `'flashrank'` uses the small cross-encoder model from [flashrank](https://github.com/PrithivirajDamodaran/FlashRank). The model reads the query and each result together. It matches meaning better. It needs `pip install flashrank`. `flashrank_scores` returns None when `flashrank` is missing. `method='flashrank'` then raises `ImportError`. `'auto'` uses `flashrank` when installed and BM25 otherwise. `'none'`, the default for `search`, keeps the merged order.

In [ ]:
#| export
_word = re.compile(r'[a-z0-9]+')
def _toks(s:str) -> list: return _word.findall((s or '').lower())

def bm25(q:str, docs:list, k1:float=1.5, b:float=0.75) -> list:
    "Return the BM25 score for each document in `docs` and query `q`. Compute word weights from `docs`."
    qt = {t for t in _toks(q) if len(t) > 1}
    dts = [_toks(d) for d in docs]
    if not (qt and dts): return [0.0]*len(docs)
    N, avg = len(dts), (sum(map(len, dts))/len(dts)) or 1.0
    idf = {}
    for t in qt:
        df = sum(1 for d in dts if t in d)
        if df: idf[t] = log(1 + (N - df + 0.5)/(df + 0.5))
    out = []
    for dt in dts:
        tf, dl, s = {}, len(dt), 0.0
        for t in dt: tf[t] = tf.get(t, 0) + 1
        for t, iv in idf.items():
            if f := tf.get(t, 0): s += iv * f*(k1+1)/(f + k1*(1 - b + b*dl/avg))
        out.append(s)
    return out

def _hit_text(h:dict) -> str:
    "Return the title, snippet and words from the URL path."
    u = h.get('href') or h.get('url') or ''
    body = h.get('body') or h.get('content') or ''
    return ' '.join((h.get('title', ''), body, re.sub(r'[/\-_.]', ' ', urlsplit(u).path)))

_ranker = None
def flashrank_scores(q:str, texts:list, model:str=None, cache_dir:str=None) -> list|None:
    "Return flashrank scores for `texts` against `q`, or None when flashrank is not installed."
    global _ranker
    try: from flashrank import Ranker, RerankRequest
    except ImportError: return None
    if _ranker is None:
        _ranker = Ranker(**({'model_name': model} if model else {}),
                         cache_dir=cache_dir or str(fossick_cache('flashrank')))
    rank = getattr(_ranker, 'rerank', None) or _ranker.rank        # Version 0.2.x renamed rank to rerank.
    out = rank(RerankRequest(query=q, passages=[dict(id=i, text=t) for i, t in enumerate(texts)]))
    s = [0.0]*len(texts)
    for d in out: s[int(d['id'])] = float(d['score'])
    return s

def _unit(xs:list) -> list:
    "Scale values to 0..1 so scorers with different scales can be combined."
    lo, hi = min(xs), max(xs)
    return [0.5]*len(xs) if hi - lo < 1e-9 else [(x-lo)/(hi-lo) for x in xs]

def rerank(q:str,               # Search query.
           hits:list,           # Fused hits from `rrf`, or any result dictionaries.
           n:int=None,          # Number of top results to keep. None keeps all.
           method:str='none',   # none, lexical, flashrank or auto.
           mix:float=0.4,       # Weight of the merged score. 1 ignores the reranker.
          ) -> list:
    "Reorder hits by relevance to `q` and the merged search score."
    if not hits or method == 'none': return hits[:n] if n else hits
    texts, rs = [_hit_text(h) for h in hits], None
    if method in ('auto', 'flashrank'):
        try: rs = flashrank_scores(q, texts)
        except Exception:
            if method == 'flashrank': raise
            rs = None                                      # Use lexical scoring when auto mode cannot use flashrank.
    if rs is None:
        if method == 'flashrank': raise ImportError('flashrank is not installed: pip install flashrank')
        rs = bm25(q, texts)
    ret = _unit([h.get('score', 0.0) for h in hits])
    rel = _unit(rs)
    out = sorted(range(len(hits)), key=lambda i: -(mix*ret[i] + (1-mix)*rel[i]))
    hits = [dict(hits[i], rerank=round(rel[i], 4)) for i in out]
    return hits[:n] if n else hits

## Where and when

The best answer to many questions depends on the country. Building rules, prices and availability differ by country. Search engines use a `country-language` region code, such as `au-en`. The default `region='auto'` reads both parts from the query.

`infer_country` looks for country names, adjectives and cities in `COUNTRY_WORDS`. It answers only when the query names one country. A query that names two countries returns None. `infer_language` counts common function words in `LANG_WORDS`, such as "the" and "what" or "der" and "und". It answers only when one language has the highest count and at least two matches. `infer_region` joins the country and language results. A query with no language signal uses the country's usual language from `COUNTRY_LANG`. The default `us-en` fills any part that the query does not define. An English question about France gives `fr-en`.

`timelimit` keeps only recent results. Use `'d'`, `'w'`, `'m'` or `'y'` for the past day, week, month or year.

In [ ]:
#| export
#: Add words that identify a country for law, price or availability queries.
COUNTRY_WORDS = {
    'au': ('australia', 'australian', 'aussie', 'nsw', 'qld', 'sydney', 'melbourne', 'brisbane',
           'canberra', 'adelaide'),
    'at': ('austria', 'austrian', 'österreich', 'oesterreich', 'wien', 'vienna', 'graz', 'salzburg'),
    'ca': ('canada', 'canadian', 'toronto', 'vancouver', 'montreal'),
    'ch': ('switzerland', 'swiss', 'schweiz', 'zürich', 'zurich', 'geneva'),
    'de': ('germany', 'german', 'deutschland', 'berlin', 'münchen', 'munich', 'hamburg'),
    'es': ('spain', 'spanish', 'españa', 'espana', 'madrid', 'barcelona'),
    'fr': ('france', 'french', 'paris', 'lyon', 'marseille'),
    'ie': ('ireland', 'irish', 'dublin'),
    'in': ('india', 'indian', 'mumbai', 'bengaluru', 'bangalore'),
    'it': ('italy', 'italian', 'italia', 'roma', 'milano', 'milan'),
    'jp': ('japan', 'japanese', 'tokyo'),
    'mx': ('mexico', 'méxico', 'mexican'),
    'nl': ('netherlands', 'dutch', 'holland', 'amsterdam', 'rotterdam'),
    'nz': ('new zealand', 'nz', 'auckland', 'wellington'),
    'br': ('brazil', 'brazilian', 'brasil'),
    'sg': ('singapore', 'singaporean'),
    'uk': ('uk', 'u.k.', 'united kingdom', 'britain', 'british', 'england', 'scotland', 'scottish',
           'wales', 'welsh', 'northern ireland', 'london', 'manchester', 'edinburgh', 'glasgow'),
    'us': ('usa', 'u.s.', 'u.s.a.', 'united states', 'america', 'american', 'new york',
           'san francisco', 'chicago'),
    'za': ('south africa', 'south african', 'johannesburg', 'cape town'),
    'ae': ('uae', 'u.a.e.', 'emirates', 'dubai', 'abu dhabi'),
}

#: Store the usual language for a country. Use it only when the query gives no language signal.
COUNTRY_LANG = {'at': 'de', 'br': 'pt', 'ch': 'de', 'de': 'de', 'es': 'es', 'fr': 'fr',
                'it': 'it', 'jp': 'jp', 'mx': 'es', 'nl': 'nl'}

#: Store function words that identify a language. Content words identify the topic instead.
LANG_WORDS = {
    'en': ('the', 'and', 'is', 'are', 'what', 'how', 'why', 'for', 'with', 'from', 'about', 'my',
           'you', 'your', 'need', 'want', 'best', 'should', 'does', 'can', 'i'),
    'de': ('der', 'die', 'das', 'und', 'ist', 'sind', 'nicht', 'für', 'mit', 'auf', 'ein', 'eine',
           'von', 'was', 'wie', 'wer', 'wo', 'muss', 'kann', 'beim', 'bei'),
    'fr': ('le', 'la', 'les', 'des', 'est', 'sont', 'pas', 'pour', 'avec', 'sur', 'une', 'quoi',
           'comment', 'pourquoi', 'dans', 'faut'),
    'es': ('el', 'los', 'las', 'y', 'es', 'son', 'no', 'para', 'con', 'sobre', 'una', 'del',
           'qué', 'cómo', 'por', 'cuál'),
    'it': ('il', 'lo', 'gli', 'è', 'sono', 'non', 'per', 'con', 'su', 'una', 'del', 'che',
           'come', 'perché', 'quale'),
    'nl': ('het', 'een', 'en', 'is', 'zijn', 'niet', 'voor', 'met', 'op', 'van', 'naar', 'wat',
           'hoe', 'waarom', 'moet'),
    'pt': ('os', 'as', 'e', 'é', 'são', 'não', 'para', 'com', 'sobre', 'uma', 'do', 'da',
           'que', 'como', 'por'),
}

#: Store `timelimit` values supported by every ddgs text engine.
TIMELIMITS = ('d', 'w', 'm', 'y')

_pats = {}
def _alt(name:str, table:dict):
    "Return one compiled alternation per table. Put the longest phrase first."
    if name not in _pats:
        pairs = sorted(((w, k) for k, ws in table.items() for w in ws), key=lambda t: -len(t[0]))
        _pats[name] = (re.compile('|'.join(f'(?<![a-z0-9])({re.escape(w)})(?![a-z0-9])'
                                           for w, _ in pairs)), [k for _, k in pairs])
    return _pats[name]

def infer_country(q:str) -> str:
    "Return the country code when `q` names one country. Return None for zero or several countries."
    rx, keys = _alt('country', COUNTRY_WORDS)
    found = {keys[m.lastindex - 1] for m in rx.finditer(ifnone(q,'').lower())}
    return found.pop() if len(found) == 1 else None

def infer_language(q:str) -> str:
    "Return the language code indicated by function words in `q`, or None when unclear."
    rx, keys = _alt('lang', LANG_WORDS)
    counts = {}
    for m in rx.finditer(ifnone(q,'').lower()): counts[keys[m.lastindex - 1]] = counts.get(keys[m.lastindex - 1], 0) + 1
    if not counts or (top := max(counts.values())) < 2: return None
    winners = [k for k, n in counts.items() if n == top]
    return winners[0] if len(winners) == 1 else None

def infer_region(q:str,             # Query used to find a place and language.
                 dflt:str='us-en',  # Default for each region part the query does not settle.
                ) -> str:
    "Return a ddgs `country-language` region from `q`. Use `dflt` for missing parts."
    dc, _, dl = (dflt or 'us-en').partition('-')
    c, lang = infer_country(q), infer_language(q)
    return f'{c or dc}-{lang or (COUNTRY_LANG.get(c, "en") if c else dl or "en")}'

def _region(q:str, region:str) -> str:
    "Resolve `region='auto'` or None from the query. Return an explicit region unchanged."
    return infer_region(q) if region in (None, 'auto') else region


## `search`

`search` is the main entry point. For text, it queries every engine in `TEXT_BACKENDS` in parallel. It merges the lists with RRF and reranks them with `method`. `backend='google'` restricts it to one engine. `n` cuts the final list only. It does not change the number of engines queried. Increase `pages` to get more candidates. With `fuse=False`, and for the `images`, `news` and `books` categories, it uses `ddgs` search for that category. The `videos` category uses `search_yt` to search YouTube.

An empty query returns an empty list without using the network. When every engine fails, `search` prints one warning per process and returns an empty list. `text`, `images`, `news`, `videos` and `books` are `search` with a category set. `extract` downloads one URL with `ddgs`. It returns `{url, content}`. The default content format is markdown.

`cache=900` keeps a result in memory for 900 seconds. A later call with the same arguments returns that result without a request. An empty result is not kept. This prevents an outage result from being kept. `fetch(cache=)` uses the same memory and the `MEM_MOST` limit. `forget()` clears the memory. The default `cache=0` asks the engines on every call.

In [ ]:
#| export
def _fused(q, n, backends, region, timeout, k, method, pages, kw=()):
    "Query backends, fuse results and rerank them. Raise when every backend fails."
    lists = _backend_lists(q, backends, region=region, timeout=timeout, pages=pages, **dict(kw))
    if not (out := rerank(q, rrf(lists, k=k), n=n, method=method)):
        raise ValueError('every backend returned nothing')
    return out

def _search_fused(q, n, backends, region, timeout, k, method, pages, kw=()):
    try: return _fused(q, n, backends, region, timeout, k, method, pages, kw)
    except ValueError as e:
        _warn('search', f'search({q!r}) got nothing: {e}')
        return []

def _search_ddgs(q, category, kw, timeout=8):
    try: return getattr(get_ddgs(timeout), category)(q, **dict(kw))
    except Exception as e:
        _warn(f'search/{category}', f'search({q!r}, {category!r}) failed: {type(e).__name__}: {e}')
        return []

def search(q:str,                  # Search query.
           category:str='text',    # text, images, news, videos or books.
           n:int=10,               # Number of results to return.
           fuse:bool=True,         # Fan out across backends and fuse with RRF for text.
           backend:str=None,       # Restrict to one ddgs backend, such as 'google'.
           method:str='none',      # Rerank method: none, lexical, flashrank or auto.
           pages:int=1,            # Result pages to pull from each backend.
           region:str='auto',      # ddgs region, or 'auto' to read the country from the query.
           timelimit:str=None,     # d, w, m or y for the last day, week, month or year.
           timeout:int=8,
           cache:float=0,          # Keep a non-empty result for this many seconds.
           **kw,                   # Extra keyword arguments passed to ddgs.
          ) -> list:
    "Search the web and merge text results from several engines with RRF."
    if not (q and q.strip()): return []
    if cache:
        a = (q, category, n, fuse, backend, method, pages, region, timelimit, timeout)
        key = repr((a, sorted(kw.items())))
        if (hs := _kept('hits', key, cache)) is not None: return hs
        return _keep('hits', key, hs) if (hs := search(*a, **kw)) else hs
    assert category in ('text', 'images', 'news', 'videos', 'books'), f'Invalid category: {category}'
    n = kw.pop('max_results', None) or n
    region = _region(q, region)
    if category == 'videos': return search_yt(q, n, **kw)
    if timelimit: kw['timelimit'] = timelimit
    if category == 'text' and fuse:
        bes = (backend,) if backend else TEXT_BACKENDS
        return _search_fused(q, n, tuple(bes), region, timeout, 60, method, max(pages, 1),
                             tuple(sorted(kw.items())))
    kw = dict(kw, max_results=n, region=region)
    if backend: kw['backend'] = backend
    return _search_ddgs(q, category, tuple(sorted(kw.items())), timeout)

text = bind(search, category='text')
images = bind(search, category='images')
news = bind(search, category='news')
videos = bind(search, category='videos')
books = bind(search, category='books')

def extract(url, **kw):
    "Download `url` with ddgs and return `{url, content}` with markdown content by default."
    return get_ddgs().extract(url, **kw)

## `google`

`google` loads Google's results page in the stealth browser and reads the results from it. The ranking is Google's own. It is slower than `search` and needs Scrapling's browsers. It accepts the same `region` and `timelimit` as `search`. The country biases the ranking. It does not exclude other countries. Google applies the time limit with its own filter. The results have the same fields as `search`. The snippet is in `content`. When Google returns no results, `google` returns an empty list. When the browser fails, it returns an empty list and prints a warning.

In [ ]:
#| export
def _parse_google(html:str, n:int) -> list:
    "Parse organic results from a rendered Google results page."
    sel = Selector(html)
    out, seen = [], set()
    for h3 in sel.css('h3'):
        a = h3.parent
        while a is not None and a.tag != 'a': a = a.parent          # Find the nearest ancestor anchor.
        if a is None: continue
        href = a.attrib.get('href', '')
        if not href.startswith('http') or 'google.' in href or href in seen: continue
        box = a
        for _ in range(4): box = box.parent if box.parent is not None else box   # Move to the result container.
        snip = first(box.css('div.VwiC3b'))                         # Read Google's snippet block.
        seen.add(href)
        out.append(dict(title=h3.get_all_text().strip(), href=href, rank=len(out)+1, engine='google',
                        content=snip.get_all_text().strip() if snip else '', score=float(n - len(out))))
        if len(out) >= n: break
    return out

def _google(q:str, n:int, lang:str, gl:str=None, tbs:str=None) -> list:
    "Fetch and parse Google results in a stealth browser. Raise on empty results."
    # `gl` biases ranking toward a country. `cr` excludes other countries.
    url = f'https://www.google.com/search?q={quote_plus(q)}&hl={lang}&num={n+2}'
    if gl: url += f'&gl={gl}'
    if tbs: url += f'&tbs={tbs}'
    pg = fetch(url, stealthy=True, network_idle=True)
    res = _parse_google(pg.html_content, n) if pg.status == 200 else []
    if not res: raise ValueError('no google results')
    return res

def _google_cached(q:str, n:int, lang:str, gl:str=None, tbs:str=None) -> list: return _google(q, n, lang, gl, tbs)

def google(q:str,                # Search query.
           n:int=10,             # Number of results.
           lang:str=None,        # `hl`; defaults to the language in `region`.
           region:str='auto',    # ddgs region, or 'auto' to read the country from the query.
           timelimit:str=None,   # d, w, m or y for Google's `tbs=qdr:` filter.
          ) -> list:
    "Search Google through the stealth browser and return results in the same shape as `search`."
    if not q or not q.strip(): return []
    gl, _, rlang = _region(q, region).partition('-')
    tbs = f'qdr:{timelimit}' if timelimit in TIMELIMITS else None
    try: return _google_cached(q, n, lang or rlang or 'en', gl, tbs)   # Cache the call that raises on an empty result.
    except ValueError: return []
    except Exception as e:
        _warn('google', f'google() is falling back to nothing: {type(e).__name__}: {e}')
        return []

## `research`

`research` turns a question into readable sources in five steps:

1. `plan`, from `fossick.quality`, splits a question with several parts into one search per part. It keeps the original question as the first search.
2. Each search uses `search`, or `google` when `engine='google'`. `research` interleaves the results by taking one result from each search in turn.
3. The default `curated=True` makes `curate` remove sites that copy other sites. It also moves authoritative sources up. `research` drops copycats before it downloads pages.
4. `research` downloads candidates in order with `fetch(auto=True)`. Each batch contains as many pages as needed to reach `n`. It stops after it gets `n` pages with at least 400 characters of readable markdown. A failed, blocked or short page goes into `dropped` with its reason. The next candidate replaces it.
5. With `focused=True`, `focus` reduces each page to about `chars` characters from passages that best answer the question.

It returns a dictionary with `query`, `sources`, `digest`, `dropped`, `region`, `curation` and `plan`. Each source contains its `title`, `href`, working fetch `tier`, publication `date`, answered `queries` and markdown in `md`. The digest joins the sources with `---`. Each source has its title, URL, publication date when known and the question part it answers.

`page_date`, `usable` and `focus` also work on their own. `page_date` finds a page's publication date and returns `YYYY-MM-DD`, or None. It reads JSON-LD data, `<meta>` tags named in `DATE_META`, a `<time>` element or a `/2018/09/` URL. `usable` checks for a 200 response, no bot wall and at least `min_chars` characters of readable markdown. `min_chars=0` skips the length check. `focus` keeps passages that answer a query. It keeps opening blocks within 30% of the character budget. It then adds the highest-scoring blocks. It returns them in their original order and uses `…` for gaps.

The next cell tests paging with stand-in engines.

In [ ]:
# Keep ranks continuous across pages and count repeated results once.
_seen = []
def _fake(be, q, region='us-en', page=1, timeout=8, tries=2, **kw):
    _seen.append((be, page))
    start = (page-1)*2
    return [dict(title=f'{be}{i}', href=f'https://e.com/{be}/{i}', body='', engine=be, rank=i-start+1)
            for i in range(start, start+3)]        # Return three hits per page and repeat one hit on the next page.
_real, _engine_hits = _engine_hits, _fake
try:
    one = _backend_lists('q', ('brave',), pages=1)
    two = _backend_lists('q', ('brave',), pages=2)
    later = _backend_lists('q', ('brave',), page=3)
finally: _engine_hits = _real

assert sorted(_seen[:3]) == [('brave',1), ('brave',1), ('brave',2)], _seen   # Run one job for each backend and page.
assert _seen[3:] == [('brave',3)]                                        # Start paging at `page`.
assert len(one[0]) == 3 and len(two[0]) == 5                             # Two pages return five unique results.
assert [h['rank'] for h in two[0]] == [1,2,3,4,5]                        # Use continuous ranks.
assert len({h['href'] for h in two[0]}) == 5                             # Remove duplicates within the backend.


In [ ]:
#| export
_MIN_CHARS = 400

#: Store lowercase `<meta>` names used for publication dates.
DATE_META = ('article:published_time', 'og:published_time', 'citation_publication_date',
             'datepublished', 'dc.date.issued', 'dcterms.date', 'sailthru.date',
             'parsely-pub-date', 'pubdate', 'publish-date', 'date')

# Put the longest alternative first. Do not allow a following digit.
_ISO = re.compile(r'(?:19|20)\d{2}[-/](?:1[0-2]|0?[1-9])(?:[-/](?:3[01]|[12]\d|0?[1-9]))?(?!\d)')

def _iso(s) -> str:
    "Return the first date in `s` as `YYYY-MM-DD`, or None when absent."
    if not s: return None
    if not (m := _ISO.search(str(s))): return None
    p = re.split(r'[-/]', m.group(0))
    return f'{p[0]}-{p[1].zfill(2)}-{(p[2].zfill(2) if len(p) > 2 else "01")}'

def _ld_nodes(html:str):
    "Return every schema.org object on the page as a flat list."
    out = []
    for m in re.finditer(r'<script[^>]+application/ld\+json[^>]*>(.*?)</script>', html, re.S | re.I):
        try: data = json.loads(m.group(1).strip())
        except Exception: continue
        todo = data if isinstance(data, list) else [data]
        while todo:
            nd = todo.pop(0)
            if not isinstance(nd, dict): continue
            out.append(nd)
            if isinstance(g := nd.get('@graph'), list): todo += g
    return out

def page_date(page) -> str:
    "Return the publication date a page states, as `YYYY-MM-DD`, or None."
    # Accept any object with markup, not only a Scrapling response.
    html = page if isinstance(page, str) else (getattr(page, 'html', None) or getattr(page, 'html_content', '') or '')
    for nd in _ld_nodes(html):
        for k in ('datePublished', 'dateCreated', 'uploadDate', 'dateModified'):
            if (d := _iso(nd.get(k))): return d
    for m in re.finditer(r'<meta[^>]+>', html, re.I):
        tag = m.group(0)
        nm = re.search(r'(?:name|property|itemprop)\s*=\s*["\']([^"\']+)', tag, re.I)
        if nm and nm.group(1).strip().lower() in DATE_META:
            c = re.search(r'content\s*=\s*["\']([^"\']+)', tag, re.I)
            if c and (d := _iso(c.group(1))): return d
    if (m := re.search(r'<time[^>]+datetime\s*=\s*["\']([^"\']+)', html, re.I)) and (d := _iso(m.group(1))): return d
    url = '' if isinstance(page, str) else str(getattr(page, 'url', '') or '')
    if (m := re.search(r'/((?:19|20)\d{2}/\d{1,2})/', url)): return _iso(m.group(1))
    return None

def _why(pg) -> str:
    "Return why a fetched page was unusable for `research()['dropped']`."
    if pg is None: return 'fetch failed'
    if (st := getattr(pg, 'status', 0)) != 200: return f'http {st}'
    if getattr(pg, 'tier', '') == 'blocked' or _blocked(pg): return 'bot wall / challenge page'
    return 'no readable content'

def usable(pg, min_chars:int=_MIN_CHARS) -> bool:
    "Return True when a page has status 200, no bot wall and enough readable markdown."
    if pg is None or getattr(pg, 'status', 0) != 200 or getattr(pg, 'tier', '') == 'blocked' or _blocked(pg): return False
    if min_chars <= 0: return True
    try: return len(html2md(getattr(pg, 'html', None) or getattr(pg, 'html_content', '') or '').strip()) >= min_chars
    except Exception: return False

def _split_blocks(md:str, max_len:int=1200) -> list:
    # Split long blocks so a block can fit within the character budget.
    "Split markdown into capped blocks and keep headings with their text."
    parts, out = [p.strip() for p in re.split(r'\n\s*\n', md) if p.strip()], []
    for p in parts:
        if out and out[-1].lstrip().startswith('#') and len(out[-1]) < max_len: out[-1] += '\n\n' + p
        else: out.append(p)
    res = []
    for p in out:
        while len(p) > max_len:
            cut = (p.rfind('. ', 0, max_len) + 1) or max_len       # Prefer a sentence boundary.
            res.append(p[:cut].strip()); p = p[cut:].strip()
        if p: res.append(p)
    return res

def focus(md:str,            # Page markdown.
          q:str,             # Query used to select relevant text.
          chars:int=4000,    # Character budget.
          reserve:float=0.3, # Fraction of the budget reserved for the page start.
          lead:float=0.6,    # Extra score for blocks near the top.
         ) -> str:
    "Keep the parts of `md` that best answer `q`, in their original order, within `chars` characters."
    if not md or len(md) <= chars: return md
    bl = _split_blocks(md)
    if len(bl) < 2: return md[:chars]
    def render(ids):
        out, prev = [], -1
        for i in sorted(ids):
            if prev >= 0 and i > prev + 1: out.append('…')
            out.append(bl[i]); prev = i
        return '\n\n'.join(out)
    keep = set()
    for i in range(len(bl)):                       # Keep the opening blocks.
        if len(render(keep | {i})) > chars*reserve: break
        keep.add(i)
    sc = bm25(q, bl)
    top = max(sc) or 1.0
    sc = [s + lead*top*(1 - i/len(bl)) for i, s in enumerate(sc)]
    for i in sorted(range(len(bl)), key=lambda i: -sc[i]):
        if i in keep or len(render(keep | {i})) > chars: continue
        keep.add(i)
        if len(render(keep)) >= chars*0.95: break
    if not keep: return max(zip(sc, bl), key=lambda t: t[0])[1][:chars]   # Return the best block when no block fits.
    return render(keep)

def _fetch_batch(urls:list, concurrency:int=8, **kw) -> list:
    "Fetch URLs in parallel and return None for raised errors."
    def _one(u):
        try: return fetch(u, **kw)
        except Exception: return None
    return list(parallel(_one, urls, n_workers=min(concurrency, len(urls) or 1), threadpool=True))

def research(q:str,                 # Search query.
             n:int=5,               # Number of readable sources to gather.
             engine:str='search',   # 'search' for fused search or 'google' for browser ranking.
             sel:str=None,          # CSS selector used before markdown conversion.
             chars:int=4000,        # Maximum markdown characters kept per page.
             auto:bool=True,        # Increase fetch methods for each page when needed.
             region:str='auto',     # ddgs region, or 'auto' to read the country from the query.
             timelimit:str=None,    # d, w, m or y for the last day, week, month or year.
             curated:bool=True,     # Drop mirror domains and rank sources before reading.
             intent:str='auto',     # Authority class, or 'auto' to read it from `q`.
             facets:int=4,          # Searches to use when `q` has several questions.
             rewrite=None,          # Optional model function to replace the planner.
             focused:bool=True,     # Keep relevant passages instead of the first `chars` characters.
             candidates:int=None,   # Number of hits to use when backfilling.
             pages:int=1,           # Result pages per backend.
             concurrency:int=8,
             **kw,                  # Extra keyword arguments passed to fetch().
            ) -> dict:
    "Search for `q`, read the top `n` readable results, and return them with a markdown digest that cites each one."
    if not (q and q.strip()):
        return dict(query=q, sources=[], digest='', dropped=[], region=_region(q, region),
                    curation=None, plan=[])
    cand = candidates or max(3*n, n+5)
    region = _region(q, region)
    qs = _plan(q, max_queries=facets, rewrite=rewrite)
    def _hits(sq):
        got = (google(sq, n=cand, region=region, timelimit=timelimit) if engine == 'google'
               else search(sq, n=cand, region=region, timelimit=timelimit, pages=pages))
        return [dict(h, queries=[sq]) for h in got]
    # Interleave facets instead of fusing them. Fusion would favour general sources.
    hits = _hits(qs[0]) if len(qs) < 2 else interleave(
        list(parallel(_hits, qs, n_workers=len(qs), threadpool=True)))
    # Curate before fetching. Candidate order selects which pages `research` reads.
    hits, curation = _curate(q, hits, intent=intent) if curated else (hits, None)
    todo, seen = [], set()
    for h in hits:
        u = h.get('href') or h.get('url')
        if u and (key := norm_url(u)) not in seen:
            seen.add(key); todo.append((h, u))
    sources, dropped, i = [], [], 0
    while len(sources) < n and i < len(todo):
        batch = todo[i:i + (n - len(sources))]; i += len(batch)
        for (h, u), pg in zip(batch, _fetch_batch([u for _, u in batch], concurrency, sel=sel, auto=auto, **kw)):
            title = h.get('title') or u
            md = ''
            if usable(pg, 0):
                try: md = to_md(pg, sel=sel)
                except Exception: md = ''
            if len(md.strip()) < _MIN_CHARS:
                dropped.append(dict(title=title, href=u, reason=_why(pg))); continue
            sources.append(dict(title=title, href=u, tier=getattr(pg, 'tier', 'plain'),
                                date=page_date(pg), queries=h.get('queries') or [q],
                                md=focus(md, q, chars) if focused else md[:chars]))
    sources = sources[:n]
    # Include the date in the digest so readers can see the publication date.
    def _head(s):
        # Include the question facet that the source answers.
        bits = [f'## {s["title"]}', s['href']]
        if s['date']: bits.append(f'published: {s["date"]}')
        if (qq := [x for x in s['queries'] if x != q]): bits.append(f'answers: {"; ".join(qq)}')
        return '\n'.join(bits)
    digest = '\n\n---\n\n'.join(f'{_head(s)}\n\n{s["md"]}' for s in sources)
    return dict(query=q, sources=sources, digest=digest, dropped=dropped, region=region,
                curation=curation, plan=qs)


In [ ]:
#: `focus` counts separators and omission markers, so the result stays within `chars`.
_page = '\n\n'.join('b%02d' % i + 'a' * 96 for i in range(40))
assert all(len(focus(_page, 'b07', chars=c)) <= c for c in (4000, 1000, 300))

## Tests

The first cells run offline. They test empty queries, the Google results parser, URL normalization, RRF, BM25, `focus`, bot wall detection, region inference, backend parameters, `page_date` and the shape of a `research` result. The later cells run real searches and are skipped by the tests. Their saved output shows what each call returns.

In [ ]:
# An empty query does nothing and uses no network.
assert search('') == []
assert research('')['sources'] == []
assert rrf([]) == [] and rerank('x', []) == []

# Parse Google results from static data without a browser or network.
_serp = ('<html><body><div class="MjjYud"><div class="g">'
         '<a href="https://ex.com/p"><h3>Example Page</h3></a>'
         '<div class="VwiC3b">A short snippet.</div></div></div>'
         '<div><a href="https://www.google.com/aclk"><h3>Ad</h3></a></div></body></html>')
_g = _parse_google(_serp, 5)
assert len(_g) == 1, f'expected 1 organic result, got {len(_g)}'   # Filter out the google.com ad link.
assert _g[0]['href'] == 'https://ex.com/p' and _g[0]['title'] == 'Example Page'
assert _g[0]['content'] == 'A short snippet.'


In [ ]:
# Treat URLs with these differences as one page.
assert norm_url('https://www.x.com/a/?utm_source=t#frag') == norm_url('http://x.com/a')
assert norm_url('https://x.com/a') != norm_url('https://x.com/b')

_g = [dict(href='https://x.com/1', rank=1, engine='google'), dict(href='https://x.com/8', rank=8, engine='google')]
_b = [dict(href='https://x.com/8', rank=8, engine='brave'),  dict(href='https://x.com/2', rank=2, engine='brave')]

# One result from each backend keeps the better-ranked result first.
_solo = rrf([_g[:1], _b[1:]])
assert [h['href'] for h in _solo] == ['https://x.com/1', 'https://x.com/2'], _solo

# Agreement across backends can outrank one strong hit.
_f = rrf([_g, _b])
assert _f[0]['href'] == 'https://x.com/8' and _f[0]['engines'] == ['google', 'brave']
assert len(_f) == 3                                                    # Remove duplicates across backends.

# Do not promote a domain without a rank.
_w = rrf([[dict(href='https://en.wikipedia.org/wiki/Rust', rank=7, engine='google'),
           dict(href='https://actix.rs/', rank=1, engine='google')]])
assert _w[0]['href'] == 'https://actix.rs/', _w


In [ ]:
# BM25 ranks the document that covers the query terms.
_docs = ['sqlite wal mode benchmark write ahead log', 'a recipe for banana bread', 'wal mode']
_s = bm25('sqlite wal mode benchmark', _docs)
assert _s[0] == max(_s) and _s[1] == 0.0, _s

# Reranking moves the relevant hit above a higher-scoring irrelevant hit.
_hits = [dict(href='https://a.com/cake', title='Banana bread', body='baking', score=1.0),
         dict(href='https://b.com/sqlite-wal-benchmark', title='SQLite WAL benchmark', body='wal vs journal', score=0.9)]
assert rerank('sqlite wal benchmark', _hits, method='lexical')[0]['href'].startswith('https://b.com')

# `focus()` keeps the passage that answers the query.
_md = ('Cookie notice. We use cookies.\n\n' + 'Navigation menu home about contact.\n\n'*8 +
       'WAL mode allows readers and writers to proceed concurrently, unlike the rollback journal.\n\n' +
       'Subscribe to our newsletter.\n\n'*8)
_f = focus(_md, 'wal mode concurrent readers writers', chars=200)
assert 'concurrently' in _f and len(_f) <= 260, _f
assert 'concurrently' not in _md[:200]      # Head truncation would miss this passage.


In [ ]:
# A Cloudflare interstitial can have status 200. Do not add it to the corpus.
class _P:
    def __init__(self, status=200, html=''): self.status, self.html_content, self.html = status, html, html
assert not usable(_P(200, '<html><head><title>Just a moment...</title></head><body>Enable JavaScript and cookies to continue</body></html>'))
assert not usable(_P(403, '<html>' + 'x'*500 + '</html>'))
assert not usable(None)
assert usable(_P(200, '<html><body>' + 'real prose about sqlite. '*40 + '</body></html>'))
assert _why(_P(429)) == 'http 429' and _why(None) == 'fetch failed'


In [ ]:
# `usable` enforces `min_chars`.
_pg = _P(200, '<html><body>' + 'short prose about sqlite. '*40 + '</body></html>')
assert usable(_pg, min_chars=100) and usable(_pg, min_chars=0)
assert not usable(_pg, min_chars=5000)

In [ ]:
# Country and language are separate answers.
assert infer_country('rebuild my garage in australia') == 'au'
assert infer_country('garage rebuild sydney quotes') == 'au'           # Cities count.
assert infer_country('planning permission in the UK') == 'uk'
assert infer_country('compare australia and germany building codes') is None   # A query naming two countries returns neither.
assert infer_country('garage rebuild cost') is None
assert infer_country('give us a quote') is None                        # Bare 'us' is the pronoun.
assert infer_country('australianism') is None                          # Match whole words.

assert infer_language('what is the best way to do this for my house') == 'en'
assert infer_language('welche Vorschriften gelten für eine Garage in Österreich') == 'de'
assert infer_language('garage') is None                                # One word is not enough evidence.
assert infer_language('la') is None                                    # This word occurs in several languages.

# Use the country from the subject and the language from the asker.
assert infer_region('I want to rebuild my garage in australia, what is the regulation') == 'au-en'
assert infer_region('garage rebuild sydney') == 'au-en'
assert infer_region('do I need planning permission for a garage in the UK') == 'uk-en'
assert infer_region('what is the cost of living in france') == 'fr-en'  # Use the French country code and English language code.
assert infer_region('welche Vorschriften gelten für eine Garage in Österreich') == 'at-de'
assert infer_region('aktuelle KI Regulierung Österreich Unternehmen 2026') == 'at-de'  # Use the country's language when no function words occur.
assert infer_region('compare australia and germany building codes') == 'us-en'         # Use the default when the country is unclear.
assert infer_region('garage rebuild cost') == 'us-en'
assert infer_region('', dflt='au-en') == 'au-en' and infer_region(None) == 'us-en'
assert _region('garage in australia', 'uk-en') == 'uk-en'              # Use the explicit region.

In [ ]:
# Pass `region` and `timelimit` to every backend.
_calls = []
def _spy(be, q, region='us-en', page=1, timeout=8, tries=2, **kw):
    _calls.append(dict(be=be, region=region, **kw))
    return [dict(title=be, href=f'https://e.com/{be}', body='', engine=be, rank=1)]
_real, _engine_hits = _engine_hits, _spy
try:
    search('sqlite wal mode', timelimit='y')
    assert [c['timelimit'] for c in _calls] == ['y']*len(TEXT_BACKENDS), _calls
    assert {c['region'] for c in _calls} == {'us-en'}

    _calls.clear(); search('rebuild my garage in australia')
    assert {c['region'] for c in _calls} == {'au-en'}, _calls
    assert not any('timelimit' in c for c in _calls)          # Omit the value so ddgs uses its default.

    _calls.clear(); search('garage in australia', region='uk-en', timelimit='m', safesearch='off')
    assert {c['region'] for c in _calls} == {'uk-en'}
    assert all(c['timelimit'] == 'm' and c['safesearch'] == 'off' for c in _calls)

    _calls.clear(); assert search('  ') == [] and _calls == []
finally: _engine_hits = _real


In [ ]:
# `cache=` keeps results for the same call. It does not keep empty results. `forget()` clears the cache.
from fossick.core import forget
_calls, _empty = [], [False]
def _spy(be, q, region='us-en', page=1, timeout=8, tries=2, **kw):
    _calls.append(be)
    return [] if _empty[0] else [dict(title=be, href=f'https://e.com/{be}', body='', engine=be, rank=1)]
_real, _engine_hits = _engine_hits, _spy
try:
    forget()
    a = search('sqlite wal mode', cache=60); asked = len(_calls)
    assert a and search('sqlite wal mode', cache=60) is a and len(_calls) == asked   # Return the cached result.
    search('sqlite wal mode', n=3, cache=60); assert len(_calls) == 2*asked          # Different arguments cause another request.
    search('sqlite wal mode'); assert len(_calls) == 3*asked                         # The default disables caching.
    forget(); search('sqlite wal mode', cache=60); assert len(_calls) == 4*asked
    _empty[0] = True; _calls.clear()
    assert search('nothing here', cache=60) == [] and search('nothing here', cache=60) == []
    assert len(_calls) == 2*asked                                                    # An empty result is not cached.
finally: _engine_hits = _real; forget()

In [ ]:
# `page_date` reads JSON-LD, `<meta>` tags, `<time>` elements and URLs.
class _D:
    def __init__(self, html='', url=''): self.html, self.url = html, url

_ld = lambda s: _D(f'<script type="application/ld+json">{s}</script>')
assert page_date(_ld('{"@type":"Article","datePublished":"2024-03-07T09:00:00Z"}')) == '2024-03-07'
assert page_date(_ld('{"@graph":[{"@type":"WebSite"},{"datePublished":"2019-11-02"}]}')) == '2019-11-02'
assert page_date(_ld('{"datePublished":"not a date","dateModified":"2024-05-05"}')) == '2024-05-05'
assert page_date(_D('<meta property="article:published_time" content="2022-06-15T10:11:12+00:00">')) == '2022-06-15'
assert page_date(_D('<meta name="DC.date.issued" content="2021-1-5">')) == '2021-01-05'
assert page_date(_D('<meta name="viewport" content="width=1"><meta name="date" content="2025-12-31">')) == '2025-12-31'
assert page_date(_D('<time datetime="2023-08-01">last August</time>')) == '2023-08-01'
assert page_date(_D('', 'https://blog.example.com/2018/09/wal-mode')) == '2018-09-01'   # Use day 01 when only the month is present.
assert page_date(_D('<p>nothing dated here</p>', 'https://e.com/x')) is None

# A malformed JSON-LD block must not hide a later meta tag.
assert page_date(_D('<script type="application/ld+json">{oops</script>'
                    '<meta name="pubdate" content="2020-02-29">')) == '2020-02-29'

# Prefer the longest month and day alternatives.
assert _iso('2019-11-02') == '2019-11-02' and _iso('2025-12-31') == '2025-12-31'
assert _iso('v1.2 build 7') is None


In [ ]:
# `research()` reports its region and keeps one result shape when no source is found.
_r = research('')
assert _r['sources'] == [] and _r['digest'] == ''
assert sorted(_r) == ['curation', 'digest', 'dropped', 'plan', 'query', 'region', 'sources']
assert _r['plan'] == []
assert _r['region'] == 'us-en' and research('', region='au-en')['region'] == 'au-en'


In [ ]:
#| eval: false
# Search text with every backend in parallel. Fuse with RRF, then rerank.
for r in search('fasthtml python web framework', n=5):
    print(f"{r['score']:.4f} {str(r.get('engines')):40s} {r['title'][:50]} — {r['href']}")

0.0320 ['brave', 'startpage']                   Python FastHTML: A Beginner’s Guide with Examples  — https://www.datacamp.com/tutorial/python-fasthtml
0.0318 ['brave', 'startpage']                   FastHTML - Modern web applications in pure Python — https://fastht.ml/
0.0315 ['brave', 'startpage']                   FastHTML: Revolutionizing Web Development with Pyt — https://www.geeksforgeeks.org/python/fasthtml-modern-web-application-in-pure-python/
0.0315 ['brave', 'startpage']                   python-fasthtml · PyPI — https://pypi.org/project/python-fasthtml/
0.0296 ['brave', 'startpage']                   GitHub - AnswerDotAI/fasthtml: The fastest way to  — https://github.com/AnswerDotAI/fasthtml


In [ ]:
#| eval: false
for r in search('red panda', 'images',n=3): print(r['title'], '—', r['image'])

Happy Red Panda — https://animalfactguide.com/wp-content/uploads/2020/12/red_panda_220-scaled.jpg
Cute Red Panda — https://i.natgeofe.com/k/63b1a8a7-0081-493e-8b53-81d01261ab5d/red-panda-full-body_16x9.jpg
Red Panda Animal Endangered Red Pandas Seized From Smuggler In Laos, — https://cincinnatizoo.org/wp-content/uploads/2012/04/Red-Panda-Image-for-Endangered-Excursion-scaled.jpg


In [ ]:
#| eval: false
# Search news.
for r in search('open source LLMs', 'news', n=3, pages=2): print(r['date'], r['title'])

2025-03-20T10:45:00+00:00 The Open-Source LLM Revolution: Transforming Enterprise AI For A New Era
2026-07-25T00:15:00+00:00 Nvidia, other tech giants caution against open-source AI ban in open letter
2023-06-20T00:00:00+00:00 Open-source AI chatbots are booming — what does this mean for researchers?


In [ ]:
#| eval: false
for r in videos(q='aditya hrudayam', n=1): print(r)

ERROR: query "aditya hrudayam" page 1: Unable to download API page: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018) (caused by CertificateVerifyError('[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)')); please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


search_yt error: ERROR: query "aditya hrudayam" page 1: Unable to download API page: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018) (caused by CertificateVerifyError('[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)')); please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


In [ ]:
#| eval: false
# Get Google's ranking through a stealth browser.
for r in google('vedicreader website', n=5): print(r['title'], '—', r['href'])

[2026-08-09 15:39:05] INFO: Fetched (200) <GET https://www.google.com/search?q=vedicreader+website&hl=en&num=7&sei=dRJ4arGzHuGihvcP0anygAk> (referer: https://www.google.com/)


VedicReader — https://vedicreader.com/
VedicReader — https://vedicreader.com/a/lgn
vedicreader.com to help absorb vedic texts easily — https://www.reddit.com/r/AdvaitaVedanta/comments/1lpr29l/vedicreadercom_to_help_absorb_vedic_texts_easily/
VedSearch — Read the Vedas in Sanskrit, Hindi & English — https://vedsearch.org/
Ved Portal - Search & Read — https://www.xn--j2b3a4c.com/en/


In [ ]:
#| eval: false
# research(): search -> read the top n results -> one cited markdown corpus
res = research('what is the raft consensus algorithm', n=3)
assert set(res) == {'query', 'sources', 'digest', 'dropped'}, res.keys()
assert res['query'] and isinstance(res['sources'], list) and res['sources']
assert all({'title','href','md','tier'} <= set(s) for s in res['sources'])
assert res['digest']
print(len(res['sources']), 'sources |', len(res['digest']), 'chars |', len(res['dropped']), 'skipped')
for d in res['dropped']: print('  skipped:', d['reason'], d['href'])

[2026-08-09 15:39:10] INFO: Fetched (200) <GET https://www.geeksforgeeks.org/system-design/raft-consensus-algorithm/> (referer: https://www.google.com/)
[2026-08-09 15:39:10] INFO: Fetched (200) <GET https://raft.github.io/> (referer: https://www.google.com/)
[2026-08-09 15:39:10] INFO: Fetched (200) <GET https://en.wikipedia.org/wiki/Raft_(algorithm)> (referer: https://www.google.com/)


3 sources | 10171 chars | 0 skipped


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()